# Inspect the current repository IPI build: PDI, PLM and address map
Fresh Bootgen/readelf inspections run directly in this notebook. This is an offline file operation, not a hardware benchmark. Enable LIVE_RUNTIME to also program this same IPI image, collect new device timings and compare them with freshly measured Python references. No eFUSE/flash writes.

In [ ]:
from pathlib import Path
import sys, os, json, time, hashlib
import pandas as pd
ROOT = next((p for base in [Path.cwd(), *Path.cwd().parents]
             for p in (base, base/'scu35') if (p/'lab.py').is_file()), None)
assert ROOT is not None, 'Open the notebook from the repository checkout'
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT/'host'))
from lab import config, tool
from notebook import runtime, show_runtime, LiveRows, command, hardware, check_build
c = config()
OUT = Path(c['build_dir']).resolve()
print('Python:', sys.executable)
print('Build:', OUT)


In [ ]:
INSPECT_BUILD = False  # True after configuring/building your repository IPI image
if INSPECT_BUILD:
    pdi = check_build(c)
    command('inspect', '--pdi', pdi, '--elf', OUT/'plm.elf', c=c)
else:
    print('Current-build inspection skipped; showing source BD address map below.')
bd = json.loads((ROOT/'hw/sys.bd').read_text())
segments = bd['design']['addressing']['/mb']['address_spaces']['Data']['segments']
display(pd.DataFrame(segments).T[['address_block','offset','range']])


In [ ]:
LIVE_RUNTIME = False
reports = runtime(live=LIVE_RUNTIME, c=c, on_line=LiveRows() if LIVE_RUNTIME else None)
comparison = show_runtime(reports)
